In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('2019.csv')
df.head()

In [ ]:
print('Shape:', df.shape)
print('---')
print('Columns:', df.columns.tolist())
print('---')
print('Missing values:')
print(df.isnull().sum())

In [ ]:
df.describe().round(2)

In [ ]:
plt.figure(figsize=(10,5))
plt.hist(df['Score'], bins=20,color='steelblue',edgecolor='white')
plt.title('Distribution of Happiness Score (2019)', fontsize=14)
plt.xlabel('Happiness Score')
plt.ylabel('Number of Countries')

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['Score'], bins=20, color='steelblue', edgecolor='white')

plt.axvline(df['Score'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["Score"].mean():.2f}')

plt.title('Distribution of Happiness Score (2019)', fontsize=14)
plt.xlabel('Happines Score')
plt.ylabel('Number of Countries')
plt.legend()

plt.show()

In [ ]:
top10 = df.sort_values('Score', ascending=False).head(10)

plt.figure(figsize=(10,5))

plt.barh(top10['Country or region'], top10['Score'], color='seagreen')

plt.gca().invert_yaxis()

plt.title('Top 10 Happiest Countries (2019)', fontsize=14)
plt.xlabel('Happiness Score')

plt.show()

In [ ]:
bottom10 = df.sort_values('Score', ascending=True).head(10)

plt.figure(figsize=(10,5))

plt.barh(bottom10['Country or region'], bottom10['Score'], color='indianred')

plt.gca().invert_yaxis()

plt.title('Top 10 Least Happy Countries (2019)', fontsize=14)
plt.xlabel('Happiness Score')

plt.show()

In [ ]:
corr = df.drop(columns=['Overall rank', 'Country or region']).corr()

plt.figure(figsize=(9, 7))

sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)

plt.title('Correlation Between Happiness Factors (2019)', fontsize=14)


plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(data=df, x='GDP per capita', y='Score', color='steelblue', s=60)

sns.regplot(data=df, x='GDP per capita', y='Score', scatter=False, color='red')

plt.title('GDP per Capita vs Happiness Score (2019)', fontsize=14)

plt.xlabel('GDP per Capita')
plt.ylabel('Happiness Score')

plt.show()

In [ ]:
slope, intercept = np.polyfit(df['GDP per capita'], df['Score'], 1)
df['Predicted'] = slope * df['GDP per capita'] + intercept
df['Residual'] = df['Score'] - df['Predicted']

print('Happier than expected:')
print(df.sort_values('Residual', ascending=False)[['Country or region', 'GDP per capita', 'Score', 'Residual']].head(5))

print('\nLess happy than expected:')
print(df.sort_values('Residual', ascending=True)[['Country or region', 'GDP per capita', 'Score', 'Residual']].head(5))

In [ ]:
outliers = pd.concat([
    df.sort_values('Residual', ascending=False).head(5),
    df.sort_values('Residual', ascending=True).head(5)
])

plt.figure(figsize=(12, 7))

sns.scatterplot(data=df, x='GDP per capita', y='Score', color='lightgray', s=60)
sns.regplot(data=df, x='GDP per capita', y='Score', scatter=False, color='red')

sns.scatterplot(data=outliers, x='GDP per capita', y='Score', 
                hue=outliers['Residual'] > 0,
               palette={True: 'seagreen', False: 'indianred'},
               s=100, legend=False)

for _, row in outliers.iterrows():
    plt.annotate(row['Country or region'],
                (row['GDP per capita'], row['Score']),
                textcoords='offset points', xytext=(6, 6), fontsize=10)

plt.title('Countries Happier / Less Happy Than Their GDP Predicts (2019)', fontsize=14)
plt.xlabel('GDP per Capita')
plt.ylabel('Happiness Score')

plt.show()

## Key Findings

**1. Most countries are moderately happy.**
Happiness scores range from 2.85 (South Sudan) to 7.77 (Finland), with a mean of ~5.41. Most countries fall between 4 and 6.

**2. The happiest countries are concentrated in Northern Europe.**
Finland, Denmark, Norway, Iceland and the Netherlands lead the ranking, while the least happy countries are mostly in Sub-Saharan Africa and conflict zones.

**3. Wealth, social support and health matter most.**
GDP per capita (r = 0.79), social support (r = 0.78) and healthy life expectancy (r = 0.78) are the strongest correlates of happiness. Generosity shows almost no relationship (r = 0.08).

**4. GDP and health are strongly linked to each other (r = 0.84).**
This makes it hard to separate their individual effects (multicollinearity), which is important for future modelling.

**5. Money is not everything.**
A simple linear regression of Score on GDP reveals clear regional patterns in the residuals:
- **Latin America** (Costa Rica, Guatemala, Nicaragua) is much happier than GDP predicts.
- **The Middle East** (Syria, Iran, Iraq, Egypt) is less happy than GDP predicts, possibly due to conflict and political instability.
- **Botswana** is the strongest negative outlier (residual -2.22).

**6. Data quality note.**
Somalia has a GDP value of exactly 0.000, which is likely missing data rather than a real value. It should be treated with caution.

*Correlation does not imply causation: these findings show associations, not proof that GDP causes happiness.*